In [11]:
# split datasets
import random

# 读取数据
words = open('names.txt', 'r').read().splitlines()

# 固定随机种子，保证每次划分结果一致
random.seed(42)
random.shuffle(words)

# 按照 80%、10%、10% 划分
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

words_train = words[:n1]
words_val = words[n1:n2]
words_test = words[n2:]

print(len(words_train), len(words_val), len(words_test))
words_train[:10], words_test[:5], words_val[:5]

25626 3203 3204


(['yuheng',
  'diondre',
  'xavien',
  'jori',
  'juanluis',
  'erandi',
  'phia',
  'samatha',
  'phoenix',
  'emmelynn'],
 ['mustafa', 'reuben', 'kahlel', 'balian', 'izael'],
 ['amay', 'aytana', 'jenevi', 'zylen', 'naim'])

## biagram sample

In [12]:
import torch
import torch.nn.functional as F

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def build_dataset(words):
    xs, ys = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']

        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])

    return torch.tensor(xs), torch.tensor(ys)


xtr, ytr = build_dataset(words_train)
xval, yval = build_dataset(words_val)
xtest, ytest = build_dataset(words_test)

In [13]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for i in range(200):
    # forward
    xenc = F.one_hot(xtr, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ytr)), ytr].log().mean()
    loss += 0.01 * (W**2).mean()

    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -100 * W.grad

print('Train loss:', loss.item())

Train loss: 2.5081064701080322


In [14]:
def evaluate_bigram(xs, ys, W):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    return loss.item()


print('Bigram model')
print('Train loss:', evaluate_bigram(xtr, ytr, W))
print('Val loss:  ', evaluate_bigram(xval, yval, W))
print('Test loss: ', evaluate_bigram(xtest, ytest, W))

Bigram model
Train loss: 2.5075314044952393
Val loss:   2.5097057819366455
Test loss:  2.513193130493164


In [15]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix = 0

    while True:
        logits = W[ix]
        counts = logits.exp()
        p = counts / counts.sum()

        ix = torch.multinomial(
            p, num_samples=1, replacement=True, generator=g
        ).item()

        out.append(itos[ix])

        if ix == 0:
            break

    print(''.join(out))

junide.
janasah.
prelay.
ainnandi.
ritolian.
juee.
ksahnaauranilevias.
dedainrwietanseiy.
elylarte.
faveumerifontume.


## Trigram sample

In [16]:
import torch
import torch.nn.functional as F

N = torch.zeros((27*27, 27), dtype=torch.int32)  # 27*27 situation for taking two characters 

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

In [17]:
def build_trigram_dataset(words):
    xs, ys = [], []

    for w in words:
        chs = ['.', '.'] + list(w) + ['.']

        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1] * 27 + stoi[ch2]
            ix2 = stoi[ch3]
            xs.append(ix1)
            ys.append(ix2)

    return torch.tensor(xs), torch.tensor(ys)


xs, ys = build_trigram_dataset(words_train)
xval3, yval3 = build_trigram_dataset(words_val)
xtest3, ytest3 = build_trigram_dataset(words_test)

num = xs.nelement()
print('number of training examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27 * 27, 27), generator=g, requires_grad=True)

number of training examples: 182625


In [18]:
# gradient descent

for i in range(200):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27*27).float()
    logits = xenc @ W

    # softmax
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # loss
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()

    # print(loss.item())
    
    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -100 * W.grad

loss.item()

2.308180332183838

In [19]:
def evaluate_trigram(xs, ys):
    xenc = F.one_hot(xs, num_classes=27 * 27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    return loss.item()


print('Trigram model')
print('Train loss:', evaluate_trigram(xs, ys))
print('Val loss:  ', evaluate_trigram(xval3, yval3))
print('Test loss: ', evaluate_trigram(xtest3, ytest3))

Trigram model
Train loss: 2.296706438064575
Val loss:   2.312896251678467
Test loss:  2.31069278717041


In [20]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix1 = 0

    while True:
        logits = W[ix1]
        counts = logits.exp()
        p = counts / counts.sum()

        ix2 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix2])

        if ix2 == 0:
            break

        ix1 = (ix1 % 27) * 27 + ix2

    print(''.join(out))

junide.
janasid.
prelay.
adin.
kai.
ritonian.
juwa.
kalania.
zabileniassdbyuinrwibel.
se.


## Result